# PHD - Audio prepocessing pipeline

## Dependencies instalation

In [11]:
pip install numpy scipy soundfile > /dev/null

Note: you may need to restart the kernel to use updated packages.


## Importing dependencies

In [33]:
"""
lfcc_py.py
Pipeline LFCC + deltas (prototipagem).
Dependências: numpy, scipy, soundfile
Como usar:
  python lfcc_py.py entrada.wav
Retorna: matriz (n_frames x n_ceps)
"""

import sys
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy.fftpack import dct
from scipy.signal import get_window

## Configurations

In [18]:
# ---------------------------
# Parâmetros (ajustáveis)
# ---------------------------
FS_TARGET = 44100
FRAME_MS = 25.0             # tamanho da janela em ms
FRAME_SHIFT_MS = 10.0       # deslocamento em ms
FFT_SIZE = 512              # pontos FFT
N_FILTERS = 24              # número de filtros (linear)
N_CEPS = 19                 # coeficientes cepstrais a manter
PREEMPH = 0.97              # pré-ênfase
DELTA_WINDOW = 2            # janela para delta

## Processing functions

In [19]:
# ---------------------------
# Funções utilitárias
# ---------------------------

def pre_emphasis(x, coef=PREEMPH):
    # Etapa 1 — pré-ênfase: y[n] = x[n] - alpha * x[n-1]
    return np.append(x[0], x[1:] - coef * x[:-1])

def framing(x, fs=FS_TARGET, frame_ms=FRAME_MS, shift_ms=FRAME_SHIFT_MS):
    # Etapa 2 — janelamento (framing)
    frame_len = int(round(frame_ms * fs / 1000.0))
    frame_step = int(round(shift_ms * fs / 1000.0))
    N = len(x)
    num_frames = 1 + max(0, int(np.ceil((N - frame_len) / frame_step)))
    pad_len = num_frames * frame_step + frame_len
    pad = np.zeros(pad_len - N)
    x_pad = np.concatenate((x, pad))
    indices = (np.tile(np.arange(0, frame_len), (num_frames,1)) +
               np.tile(np.arange(0, num_frames * frame_step, frame_step), (frame_len,1)).T)
    frames = x_pad[indices.astype(np.int32, copy=False)]
    # janela Hamming
    win = get_window("hamming", frame_len, fftbins=True)
    return frames * win

def stft_power(frames, NFFT=FFT_SIZE):
    # Etapa 3 & 4 — FFT + espectro de potência
    # Retorna P[k] (apenas metade positiva, 0..NFFT/2)
    complex_spec = np.fft.rfft(frames, n=NFFT)
    power_spec = (1.0 / NFFT) * (np.abs(complex_spec) ** 2)  # energia por bin
    return power_spec  # shape: (n_frames, NFFT/2 + 1)

def linear_filterbank(n_fft, fs=FS_TARGET, n_filters=N_FILTERS, fmin=0.0, fmax=None):
    # Etapa 5 — construção do banco de filtros triangulares linearmente espaçados
    if fmax is None:
        fmax = fs / 2.0
    # centros em escala linear entre fmin e fmax
    centers = np.linspace(fmin, fmax, n_filters + 2)  # includes edges
    # converter frequências para índices de bin da FFT
    bins = np.floor((n_fft + 1) * centers / fs).astype(int)
    fb = np.zeros((n_filters, n_fft // 2 + 1))
    for m in range(1, n_filters + 1):
        f_m_minus = bins[m - 1]
        f_m = bins[m]
        f_m_plus = bins[m + 1]
        # subida linear
        if f_m > f_m_minus:
            fb[m - 1, f_m_minus:f_m] = np.linspace(0, 1, f_m - f_m_minus, endpoint=False)
        # descida linear
        if f_m_plus > f_m:
            fb[m - 1, f_m:f_m_plus] = np.linspace(1, 0, f_m_plus - f_m, endpoint=False)
    return fb, centers

def lifter(cepstra, L=0):
    # opcional: liftering (não obrigatório); L>0 aplica
    if L > 0:
        nframes, ncoeff = cepstra.shape
        n = np.arange(ncoeff)
        lift = 1 + (L / 2.0) * np.sin(np.pi * n / L)
        return cepstra * lift
    return cepstra

def compute_deltas(feat, N=DELTA_WINDOW):
    # Cálculo de delta (derivada temporal)
    denom = 2 * sum([i*i for i in range(1, N+1)])
    padded = np.pad(feat, ((N,N),(0,0)), mode='edge')
    deltas = np.zeros_like(feat)
    for t in range(feat.shape[0]):
        num = np.zeros(feat.shape[1])
        for n in range(1, N+1):
            num += n * (padded[t+N+n] - padded[t+N-n])
        deltas[t] = num / denom
    return deltas

## Main pipeline

In [20]:
# ---------------------------
# Pipeline principal
# ---------------------------

def lfcc_from_file(wav_path):
    x, fs = sf.read(wav_path)
    # se estéreo, tomar média (mono)
    if x.ndim > 1:
        x = x.mean(axis=1)
    # — opcional: reamostrar se necessário (aqui assumimos já em FS_TARGET)
    if fs != FS_TARGET:
        raise RuntimeError("Amostragem diferente de {} Hz; reamostrar antes.".format(FS_TARGET))
    # 1) pré-ênfase
    x_pre = pre_emphasis(x)
    # 2) framing + janela
    frames = framing(x_pre, fs=fs)
    # 3-4) FFT + potência
    P = stft_power(frames, NFFT=FFT_SIZE)
    # 5) banco linear
    fb, centers = linear_filterbank(FFT_SIZE, fs=fs, n_filters=N_FILTERS)
    # 6) energia por banda (matmul: frames x bins * fb.T)
    # shape: (n_frames, n_filters)
    energy = np.dot(P, fb.T)
    # numerical stability: floor tiny values
    energy[energy < 1e-12] = 1e-12
    # 7) log
    log_energy = np.log(energy)
    # 8) DCT -> cepstra
    cepstra = dct(log_energy, type=2, axis=1, norm='ortho')[:, :N_CEPS]
    # opcional: lifter (não usado aqui)
    cepstra = lifter(cepstra, L=0)
    # 9) deltas e deltadeltas (opcional)
    delta = compute_deltas(cepstra)
    delta2 = compute_deltas(delta)
    # concatenar [static | delta | delta2]
    feat = np.concatenate([cepstra, delta, delta2], axis=1)
    return feat, centers

## Execute pipeline

In [34]:
feats, centers = lfcc_from_file("/home/ensismoebius/Músicas/lala.wav")
print("LFCC features shape:", feats.shape)
# salva em .npy
np.save("lfcc_feats.npy", feats)
print("Saved lfcc_feats.npy")

LFCC features shape: (1121, 57)
Saved lfcc_feats.npy
